# Tabular Ensemble

`XGBoost + LightGBM`? ? ????? ?? ????, ? ??? `split + global` ???? ?? ?? ? ???? ?????.

XGBoost ????? ?? `XGBoost.ipynb`? ???? ?????.

In [17]:
from pathlib import Path

import itertools
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

In [18]:
DATA_DIR = Path('../data/processed_data')
TRAIN_PATH = DATA_DIR / 'train_processed.parquet'
VALID_PATH = DATA_DIR / 'valid_processed.parquet'
OUTPUT_PATH = Path('./tabular_ensemble_valid_predictions.csv')

TARGET_COL = 'target'
ROW_ID_COL = 'row_id'
CAT_COLS = ['county', 'is_business', 'product_type', 'is_consumption', 'segment']
DROP_COLS = {TARGET_COL, ROW_ID_COL, 'datetime', 'date'}

train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)

for col in CAT_COLS:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('category')
        valid_df[col] = valid_df[col].astype('category')

FEATURE_COLS_ALL = [col for col in train_df.columns if col not in DROP_COLS]
FEATURE_COLS_SPLIT = [col for col in FEATURE_COLS_ALL if col != 'is_consumption']

print('train shape:', train_df.shape)
print('valid shape:', valid_df.shape)
print('feature count (all):', len(FEATURE_COLS_ALL))
print('feature count (split):', len(FEATURE_COLS_SPLIT))

train shape: (1614612, 140)
valid shape: (403740, 140)
feature count (all): 136
feature count (split): 135


In [19]:
def split_by_consumption(df):
    cons = df[df['is_consumption'] == 1].copy()
    prod = df[df['is_consumption'] == 0].copy()
    return cons, prod


def align_prediction_frames(pred_split_df, pred_all_df, pred_col):
    pred_split_df = pred_split_df.sort_values(ROW_ID_COL).reset_index(drop=True)
    pred_all_df = pred_all_df.sort_values(ROW_ID_COL).set_index(ROW_ID_COL)
    pred_all_aligned = pred_all_df.loc[pred_split_df[ROW_ID_COL]].reset_index()
    pred_split_df[f'{pred_col}_ensemble'] = 0.5 * pred_split_df[pred_col].values + 0.5 * pred_all_aligned[pred_col].values
    return pred_split_df


def summarize_segment_mae(mae_prod, mae_cons, mae_split, mae_all, mae_ensemble, model_name):
    return pd.DataFrame(
        {'MAE': [mae_prod, mae_cons, mae_split, mae_all, mae_ensemble]},
        index=[
            f'{model_name} production',
            f'{model_name} consumption',
            f'{model_name} overall(split)',
            f'{model_name} overall(single)',
            f'{model_name} split-ensemble',
        ],
    )


def train_split_global(train_df, valid_df, feature_cols_split, feature_cols_all, trainer, pred_col, model_name):
    tr_cons, tr_prod = split_by_consumption(train_df)
    va_cons, va_prod = split_by_consumption(valid_df)

    model_prod, pred_prod_df, mae_prod = trainer(tr_prod, va_prod, feature_cols_split, 'production')
    model_cons, pred_cons_df, mae_cons = trainer(tr_cons, va_cons, feature_cols_split, 'consumption')
    model_all, pred_all_df, mae_all = trainer(train_df, valid_df, feature_cols_all, 'overall')

    pred_split_df = pd.concat([pred_prod_df, pred_cons_df], axis=0).sort_values(ROW_ID_COL).reset_index(drop=True)
    mae_split = mean_absolute_error(pred_split_df[TARGET_COL], pred_split_df[pred_col])

    pred_split_df = align_prediction_frames(pred_split_df, pred_all_df, pred_col)
    ensemble_col = f'{pred_col}_ensemble'
    mae_ensemble = mean_absolute_error(pred_split_df[TARGET_COL], pred_split_df[ensemble_col])

    summary = summarize_segment_mae(mae_prod, mae_cons, mae_split, mae_all, mae_ensemble, model_name)
    display(summary.style.format('{:.4f}'))

    return {
        'model_prod': model_prod,
        'model_cons': model_cons,
        'model_all': model_all,
        'pred_split_df': pred_split_df,
        'pred_all_df': pred_all_df,
        'summary': summary,
        'mae_prod': mae_prod,
        'mae_cons': mae_cons,
        'mae_split': mae_split,
        'mae_all': mae_all,
        'mae_ensemble': mae_ensemble,
        'pred_col': pred_col,
        'ensemble_col': ensemble_col,
    }

In [20]:
XGB_BASE_PARAMS = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'device': 'cuda',
    'enable_categorical': True,
    'random_state': 42,
    'early_stopping_rounds': 50,
    'eval_metric': 'mae',
    'verbosity': 0,
}


def train_xgb_segment(train_part, valid_part, feature_cols, segment_name):
    params = XGB_BASE_PARAMS.copy()
    model = xgb.XGBRegressor(**params)

    X_train = train_part[feature_cols].fillna(0)
    y_train = train_part[TARGET_COL]
    X_valid = valid_part[feature_cols].fillna(0)
    y_valid = valid_part[TARGET_COL]

    try:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)
    except xgb.core.XGBoostError as exc:
        print(f'[XGBoost:{segment_name}] CUDA unavailable, falling back to CPU: {exc}')
        params['device'] = 'cpu'
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)

    pred = np.clip(model.predict(X_valid), 0, None)
    pred_df = valid_part[[ROW_ID_COL, TARGET_COL]].copy()
    pred_df['pred_xgb'] = pred
    mae = mean_absolute_error(y_valid, pred)
    best_iter = getattr(model, 'best_iteration', None)
    print(f'[XGBoost:{segment_name}] MAE = {mae:.4f} | Best iter: {best_iter}')
    return model, pred_df, mae

In [21]:
LGBM_BASE_PARAMS = {
    'objective': 'regression',
    'learning_rate': 0.04,
    'n_estimators': 1800,
    'num_leaves': 127,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'random_state': 42,
    'device_type': 'gpu',
}


def train_lgbm_segment(train_part, valid_part, feature_cols, segment_name):
    params = LGBM_BASE_PARAMS.copy()
    model = lgb.LGBMRegressor(**params)

    X_train = train_part[feature_cols].copy().fillna(0)
    y_train = train_part[TARGET_COL].copy()
    X_valid = valid_part[feature_cols].copy().fillna(0)
    y_valid = valid_part[TARGET_COL].copy()
    categorical_cols = [col for col in CAT_COLS if col in feature_cols]

    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric='mae',
            categorical_feature=categorical_cols,
            callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
        )
    except Exception as exc:
        print(f'[LightGBM:{segment_name}] GPU unavailable, falling back to CPU: {exc}')
        params['device_type'] = 'cpu'
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric='mae',
            categorical_feature=categorical_cols,
            callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
        )

    pred = np.clip(model.predict(X_valid), 0, None)
    pred_df = valid_part[[ROW_ID_COL, TARGET_COL]].copy()
    pred_df['pred_lgbm'] = pred
    mae = mean_absolute_error(y_valid, pred)
    print(f'[LightGBM:{segment_name}] MAE = {mae:.4f}')
    return model, pred_df, mae

In [22]:
xgb_result = train_split_global(
    train_df,
    valid_df,
    FEATURE_COLS_SPLIT,
    FEATURE_COLS_ALL,
    train_xgb_segment,
    'pred_xgb',
    'XGBoost',
)

[0]	validation_0-mae:177.55114
[100]	validation_0-mae:21.62312
[200]	validation_0-mae:19.41310
[300]	validation_0-mae:18.48567
[400]	validation_0-mae:18.03963
[500]	validation_0-mae:17.73265
[600]	validation_0-mae:17.51640
[700]	validation_0-mae:17.33847
[800]	validation_0-mae:17.21289
[900]	validation_0-mae:17.10636
[999]	validation_0-mae:17.01275
[XGBoost:production] MAE = 16.9639 | Best iter: 999
[0]	validation_0-mae:529.03953
[100]	validation_0-mae:25.96471
[200]	validation_0-mae:20.59153
[300]	validation_0-mae:18.90834
[400]	validation_0-mae:18.16539
[500]	validation_0-mae:17.65169
[600]	validation_0-mae:17.27250
[700]	validation_0-mae:17.01086
[800]	validation_0-mae:16.81746
[900]	validation_0-mae:16.58117
[999]	validation_0-mae:16.43831
[XGBoost:consumption] MAE = 16.4323 | Best iter: 999
[0]	validation_0-mae:378.07245
[100]	validation_0-mae:24.04049
[200]	validation_0-mae:19.68217
[300]	validation_0-mae:18.23612
[400]	validation_0-mae:17.26912
[500]	validation_0-mae:16.66436
[6

,MAE
XGBoost production,16.9639
XGBoost consumption,16.4323
XGBoost overall(split),16.6981
XGBoost overall(single),15.1330
XGBoost split-ensemble,14.1210


In [23]:
lgbm_result = train_split_global(
    train_df,
    valid_df,
    FEATURE_COLS_SPLIT,
    FEATURE_COLS_ALL,
    train_lgbm_segment,
    'pred_lgbm',
    'LightGBM',
)

[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 25707
[LightGBM] [Info] Number of data points in the train set: 807306, number of used features: 133
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4060 Ti, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 122 dense feature groups (95.47 MB) transferred to GPU in 0.029263 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 71.726586
Training until validation scores don't improve for 120 rounds
[100]	valid_0's l1: 21.225	valid_0's l2: 27296
[200]	valid_0's l1: 17.9808	valid_0's l2: 25336
[300]	valid_0's l1: 17.2832	valid_0's l2: 25214.7
[400]	valid_0's l1: 16.9495	valid_0's l2: 25164
[500]	valid_0's l1: 16.7626	valid_0's l2: 25170.7
Early stopping, best iteration is:
[417]	valid_0's l1: 16.9067	valid_0's l2: 25153.2
[LightGBM:production]

,MAE
LightGBM production,16.8920
LightGBM consumption,17.3757
LightGBM overall(split),17.1339
LightGBM overall(single),16.5610
LightGBM split-ensemble,14.9584


In [24]:
ensemble_df = (
    xgb_result['pred_split_df'][[ROW_ID_COL, TARGET_COL, xgb_result['ensemble_col']]]
    .rename(columns={xgb_result['ensemble_col']: 'pred_xgb'})
    .merge(
        lgbm_result['pred_split_df'][[ROW_ID_COL, lgbm_result['ensemble_col']]].rename(columns={lgbm_result['ensemble_col']: 'pred_lgbm'}),
        on=ROW_ID_COL,
        how='inner',
    )
    .sort_values(ROW_ID_COL)
    .reset_index(drop=True)
)

ensemble_df['pred_simple_avg'] = ensemble_df[['pred_xgb', 'pred_lgbm']].mean(axis=1)
simple_avg_mae = mean_absolute_error(ensemble_df[TARGET_COL], ensemble_df['pred_simple_avg'])
print('ensemble rows:', len(ensemble_df))
print(f'Simple average MAE: {simple_avg_mae:.4f}')
ensemble_df.head()

ensemble rows: 403740
Simple average MAE: 14.0380


,row_id,target,pred_xgb,pred_lgbm,pred_simple_avg
0,1614612,0.005,0.848618,0.601346,0.724982
1,1614613,726.559,735.735962,733.052449,734.394206
2,1614614,0.000,0.000000,0.000000,0.000000
3,1614615,29.203,29.710060,28.426876,29.068468
4,1614616,1.175,0.941151,0.797778,0.869464


In [25]:
weight_rows = []
weights = np.round(np.arange(0.0, 1.01, 0.05), 2)

for wx in weights:
    wl = round(1.0 - wx, 2)
    pred = wx * ensemble_df['pred_xgb'] + wl * ensemble_df['pred_lgbm']
    mae = mean_absolute_error(ensemble_df[TARGET_COL], pred)
    weight_rows.append({
        'weight_xgb': wx,
        'weight_lgbm': wl,
        'mae': mae,
    })

weight_table = pd.DataFrame(weight_rows).sort_values('mae').reset_index(drop=True)
display(weight_table.style.format({
    'weight_xgb': '{:.2f}',
    'weight_lgbm': '{:.2f}',
    'mae': '{:.4f}',
}))

best_weights = weight_table.iloc[0]
print('Best weights:')
print(best_weights)

,weight_xgb,weight_lgbm,mae
0,0.70,0.30,13.9457
1,0.75,0.25,13.9488
2,0.65,0.35,13.9532
3,0.80,0.20,13.9624
4,0.60,0.40,13.9710
5,0.85,0.15,13.9866
6,0.55,0.45,13.9992
7,0.90,0.10,14.0208
8,0.50,0.50,14.0380
9,0.95,0.05,14.0658


Best weights:
weight_xgb      0.700000
weight_lgbm     0.300000
mae            13.945684
Name: 0, dtype: float64


In [26]:
best_pred = (
    best_weights['weight_xgb'] * ensemble_df['pred_xgb']
    + best_weights['weight_lgbm'] * ensemble_df['pred_lgbm']
)
ensemble_df['pred_best_weighted'] = best_pred
best_weighted_mae = mean_absolute_error(ensemble_df[TARGET_COL], ensemble_df['pred_best_weighted'])

result_summary = pd.DataFrame(
    {
        'MAE': [
            xgb_result['mae_ensemble'],
            lgbm_result['mae_ensemble'],
            simple_avg_mae,
            best_weighted_mae,
        ]
    },
    index=[
        'XGBoost split+global',
        'LightGBM split+global',
        'Simple average (2 models)',
        'Best weighted ensemble',
    ],
)

display(result_summary.style.format('{:.4f}'))
ensemble_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved validation predictions to: {OUTPUT_PATH.resolve()}')

,MAE
XGBoost split+global,14.1210
LightGBM split+global,14.9584
Simple average (2 models),14.0380
Best weighted ensemble,13.9457


Saved validation predictions to: C:\Users\AN\Desktop\kaggle\notebooks\tabular_ensemble_valid_predictions.csv
